# Lens I — Relational Distance Structure

**Geometria Iuris** — Tesi magistrale in Metodologia delle Scienze Giuridiche (LUISS)

This notebook demonstrates the analysis pipeline for Lens I, which examines whether legal embedding spaces
preserve domain-internal relational structure and whether that structure is shared across linguistic traditions.

## Thesis sections covered

| Section | Content |
|---|---|
| §3.1.1 | Background term domain assignment via k-NN projection |
| §3.1.1 | Domain signal tests: intra vs. inter-domain, legal vs. control |
| §3.1.2 | Domain topology K×K |
| §3.1.4 | Within-tradition and cross-tradition RSA |

**Inference discipline** (§ *Disciplina inferenziale* in CLAUDE.md):
All figures in this notebook are *measures*. Juridical interpretations are labelled
explicitly and are always accompanied by their epistemic limits.

In [1]:
import sys
from pathlib import Path
import numpy as np
import json

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from shared.embeddings import load_precomputed
from shared.statistical import compute_rdm, upper_tri, mannwhitney_with_r, rsa
from lens_1_relational.domain_assignment import assign_domains

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
RESULTS_DIR = ROOT / "lens_1_relational" / "results"
WEIRD_LABELS = ["BGE-EN-large", "E5-large", "FreeLaw-EN"]
SINIC_LABELS = ["BGE-ZH-large", "Text2vec-large-ZH", "Dmeta-ZH"]

## Load data

In [2]:
_, index = load_precomputed("BGE-EN-large", EMB_DIR)
core_idx  = [i for i, t in enumerate(index) if t["tier"] == "core" and t["domain"]]
bg_idx    = [i for i, t in enumerate(index) if t["tier"] == "background"]
ctrl_idx  = [i for i, t in enumerate(index) if t["tier"] == "control"]
terms_core = [index[i] for i in core_idx]
domains = [t["domain"] for t in terms_core]
print(f"core={len(core_idx)}  background={len(bg_idx)}  control={len(ctrl_idx)}")
from collections import Counter
print("Domain distribution:", dict(sorted(Counter(domains).items())))

core=397  background=8975  control=100
Domain distribution: {'administrative': 12, 'civil': 136, 'constitutional': 48, 'criminal': 66, 'international': 52, 'labor_social': 30, 'procedure': 53}


## §3.1.1 — Background term domain assignment (k-NN, k=7)

In [3]:
vecs, _ = load_precomputed("BGE-EN-large", EMB_DIR)
vecs_core = vecs[core_idx]
vecs_bg   = vecs[bg_idx]
labels_core = [t["domain"] for t in terms_core]

assignments = assign_domains(vecs_bg, vecs_core, labels_core, k=7)

confidences = np.array([a["confidence"] for a in assignments])
from collections import Counter
domain_counts = Counter(a["assigned_domain"] for a in assignments)
print("Assigned domain distribution:", dict(sorted(domain_counts.items())))
print(f"Confidence: mean={confidences.mean():.3f}  median={np.median(confidences):.3f}")
print(f"Low confidence (<4/7\u22480.57): {(confidences < 4/7).sum()} terms ({100*(confidences < 4/7).mean():.1f}%)")

Assigned domain distribution: {'administrative': 31, 'civil': 4412, 'constitutional': 720, 'criminal': 1505, 'international': 706, 'labor_social': 665, 'procedure': 936}
Confidence: mean=0.524  median=0.429
Low confidence (<4/7≈0.57): 6923 terms (77.1%)


In [4]:
terms_bg = [index[i] for i in bg_idx]
sorted_by_conf = sorted(zip(terms_bg, assignments), key=lambda x: x[1]["confidence"], reverse=True)

print("=== 5 HIGH-CONFIDENCE assignments ===")
for term, asgn in sorted_by_conf[:5]:
    print(f"  [{asgn['confidence']:.2f}] {term['en']!r:40s} \u2192 {asgn['assigned_domain']}")
    for j in range(3):
        nb = terms_core[asgn['neighbor_indices'][j]]
        print(f"         nb{j+1}: {nb['en']!r:35s} [{asgn['neighbor_domains'][j]}] sim={asgn['neighbor_sims'][j]:.3f}")

print("\n=== 5 LOW-CONFIDENCE assignments (ambiguous) ===")
low_conf = [(t, a) for t, a in zip(terms_bg, assignments) if a["confidence"] < 4/7]
for term, asgn in sorted(low_conf, key=lambda x: x[1]["confidence"])[:5]:
    print(f"  [{asgn['confidence']:.2f}] {term['en']!r:40s} \u2192 {asgn['assigned_domain']}")
    from collections import Counter
    vote_summary = Counter(asgn["neighbor_domains"])
    print(f"         votes: {dict(vote_summary)}")

=== 5 HIGH-CONFIDENCE assignments ===
  [1.00] 'abeyant property'                       → civil
         nb1: 'ex-tenant'                         [civil] sim=0.727
         nb2: 'leasehold'                         [civil] sim=0.709
         nb3: 'tenant'                            [civil] sim=0.708
  [1.00] 'abode'                                  → civil
         nb1: 'tenant'                            [civil] sim=0.754
         nb2: 'landlord'                          [civil] sim=0.725
         nb3: 'co-tenant'                         [civil] sim=0.720
  [1.00] 'aliunde'                                → civil
         nb1: 'pledgee'                           [civil] sim=0.673
         nb2: 'testate'                           [civil] sim=0.635
         nb3: 'bailment'                          [civil] sim=0.622
  [1.00] 'amenity'                                → civil
         nb1: 'tenantable'                        [civil] sim=0.699
         nb2: 'tenant'                            

## §3.1.1 — Domain signal tests (WEIRD models)
### §3.1.1 Intra-domain vs inter-domain distances

In [5]:
from lens_1_relational.lens1 import _intra_inter_split

results_821 = {}
for label in WEIRD_LABELS:
    vecs, _ = load_precomputed(label, EMB_DIR)
    vecs_core = vecs[core_idx]
    rdm = compute_rdm(vecs_core)
    intra, inter = _intra_inter_split(rdm, domains)
    res = mannwhitney_with_r(intra, inter, alternative="less")
    results_821[label] = res
    print(f"{label:20s}  intra_med={res.median_x:.3f}  inter_med={res.median_y:.3f}  r={res.effect_r:+.3f}  p={res.p_value:.2e}")

BGE-EN-large          intra_med=0.436  inter_med=0.465  r=+0.284  p=0.00e+00
E5-large              intra_med=0.224  inter_med=0.234  r=+0.233  p=0.00e+00
FreeLaw-EN            intra_med=0.603  inter_med=0.640  r=+0.298  p=0.00e+00


### §3.1.1 Legal vs control signal

In [6]:
results_822 = {}
for label in WEIRD_LABELS:
    vecs, _ = load_precomputed(label, EMB_DIR)
    vecs_core_l = vecs[core_idx]
    vecs_ctrl   = vecs[ctrl_idx]
    n_core = len(core_idx)
    from shared.statistical import compute_rdm, upper_tri
    combined = np.vstack([vecs_core_l, vecs_ctrl])
    rdm_lc = compute_rdm(combined)
    legal_legal = upper_tri(rdm_lc[:n_core, :n_core])
    legal_ctrl  = rdm_lc[:n_core, n_core:].flatten()
    res = mannwhitney_with_r(legal_legal, legal_ctrl, alternative="less")
    results_822[label] = res
    print(f"{label:20s}  legal_med={res.median_x:.3f}  ctrl_med={res.median_y:.3f}  r={res.effect_r:+.3f}  p={res.p_value:.2e}")

BGE-EN-large          legal_med=0.460  ctrl_med=0.468  r=+0.101  p=1.61e-176
E5-large              legal_med=0.232  ctrl_med=0.241  r=+0.223  p=0.00e+00
FreeLaw-EN            legal_med=0.633  ctrl_med=0.612  r=-0.174  p=1.00e+00


### §3.1.2 Domain topology K×K

In [7]:
from lens_1_relational.lens1 import _domain_topology

vecs, _ = load_precomputed("BGE-EN-large", EMB_DIR)
rdm_core = compute_rdm(vecs[core_idx])
domain_labels, topo = _domain_topology(rdm_core, domains)

print("Domain topology (mean cosine distance) \u2014 BGE-EN-large")
header = f"{'':22s}" + "".join(f"{d[:8]:>10s}" for d in domain_labels)
print(header)
for i, d1 in enumerate(domain_labels):
    row = f"{d1:22s}" + "".join(f"{topo[i,j]:10.3f}" for j in range(len(domain_labels)))
    print(row)

Domain topology (mean cosine distance) — BGE-EN-large
                        administ     civil  constitu  criminal  internat  labor_so  procedur
administrative             0.473     0.490     0.473     0.497     0.489     0.481     0.482
civil                      0.490     0.441     0.464     0.458     0.470     0.476     0.449
constitutional             0.473     0.464     0.418     0.459     0.455     0.476     0.459
criminal                   0.497     0.458     0.459     0.396     0.478     0.477     0.444
international              0.489     0.470     0.455     0.478     0.424     0.488     0.461
labor_social               0.481     0.476     0.476     0.477     0.488     0.419     0.474
procedure                  0.482     0.449     0.459     0.444     0.461     0.474     0.417


## §3.1.4 — RSA: Within-tradition and Cross-tradition

In [ ]:
results_path = RESULTS_DIR / "lens1_results.json"
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

    rsa_data = results.get("section_314", {})

    print("=== Within-WEIRD RSA (§3.1.4) ===")
    for r in rsa_data.get("within_weird", []):
        print(f"  {r['model_a']:20s} × {r['model_b']:20s}  ρ={r['rho']:+.3f}  r²={r['r_squared']:.3f}  CI=[{r['ci_low']:.3f},{r['ci_high']:.3f}]  p={r['p_value']:.3f}")

    print("\n=== Within-Sinic RSA (§3.1.4) ===")
    for r in rsa_data.get("within_sinic", []):
        print(f"  {r['model_a']:20s} × {r['model_b']:20s}  ρ={r['rho']:+.3f}  r²={r['r_squared']:.3f}  CI=[{r['ci_low']:.3f},{r['ci_high']:.3f}]  p={r['p_value']:.3f}")

    print("\n=== Cross-tradition RSA (§3.1.4) ===")
    for r in rsa_data.get("cross_tradition", []):
        print(f"  {r['model_a']:15s} × {r['model_b']:20s}  ρ={r['rho']:+.3f}  r²={r['r_squared']:.3f}  CI=[{r['ci_low']:.3f},{r['ci_high']:.3f}]  p={r['p_value']:.3f}")

    summary = rsa_data.get("summary", {})
    print(f"\n  ρ̄ within-WEIRD={summary.get('mean_rho_within_weird','?')}  "
          f"within-Sinic={summary.get('mean_rho_within_sinic','?')}  "
          f"cross={summary.get('mean_rho_cross','?')}")
    print(f"  Cross-tradition drop: Δρ = {summary.get('cross_tradition_drop','?')}")
else:
    print("Results not found. Run first:")
    print("  cd experiments/")
    print("  python -m lens_1_relational.lens1")